# Stage 1 Ablation Runner (M7B)

Runs **one** ablation preset end-to-end on a Colab T4 and saves its artefacts to `training/runs/<preset_name>/` for later cross-run analysis in `notebooks/analysis.ipynb`.

**Three presets** (defined in `training/ablations.py`):

| name | curriculum | aux reward | what it tests |
|---|---|---|---|
| `baseline_replay` | on  | off (0.0)  | reproduce Stage 1 for fair comparison |
| `no_curriculum`   | off | off (0.0)  | does the easy→hard schedule actually help? |
| `aux_direction`   | on  | on  (0.10) | does paying for signal-aligned trades fix Probe 3? |

**Time budget per preset (T4):** ~25 min SFT + ~1.5 hr GRPO (200 steps) + ~25 min eval = **~2.5 hr**.

**Recommended order**: run `aux_direction` first (it's the most diagnostic — Stage 1 already shows curriculum-on as the strong baseline). If you have time after that, run `no_curriculum`.

## 1. Install dependencies

Pinned to the same versions used by `train_colab.ipynb`.

In [ ]:
%%capture
# Let Unsloth resolve its own deps (xformers, unsloth_zoo, triton, ...) so the
# version matches whatever torch the current Colab image ships with. The old
# xformers==0.0.27 pin from Stage 1 is removed because it forced a torch
# downgrade that broke the kernel on Colab T4 (Apr 2026).
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets sentencepiece protobuf wandb

## 2. Clone the repo + pick a preset

Change `PRESET_NAME` to the ablation you want.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/PrathameshWable/market-rl-env'
REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# >>> EDIT THIS LINE <<<
PRESET_NAME = 'aux_direction'   # one of: baseline_replay, no_curriculum, aux_direction

from training.ablations import get_preset, make_difficulty_scheduler
CFG = get_preset(PRESET_NAME)
print(f'Preset:   {CFG.name}')
print(f'Curriculum: {CFG.use_curriculum}')
print(f'Aux reward weight: {CFG.aux_direction_weight}')
print(f'SFT steps: {CFG.sft_steps}, GRPO steps: {CFG.grpo_steps}')
print(f'Description: {CFG.description}')

## 3. Mount Drive, optional W&B

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_RUN_DIR = f'/content/drive/MyDrive/market-rl-ablations/{CFG.name}'
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

WANDB_ENABLED = False
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_PROJECT'] = 'market-rl-ablations'
    os.environ['WANDB_NAME'] = CFG.name
    WANDB_ENABLED = True
    print('W&B enabled')
except Exception:
    print('No WANDB_API_KEY — skipping W&B logging.')

## 4. Reuse SFT data (or regenerate)

If you already have `training/sft_data.jsonl` from Stage 1 in Drive, copy it in to skip the 7-minute generation. Otherwise it'll be regenerated.

In [ ]:
from pathlib import Path

SFT_PATH = Path('training/sft_data.jsonl')
DRIVE_SFT_PATH = Path('/content/drive/MyDrive/market-rl-stage1/sft_data.jsonl')

if not SFT_PATH.exists() and DRIVE_SFT_PATH.exists():
    print(f'Reusing cached SFT data from {DRIVE_SFT_PATH}')
    import shutil; shutil.copy(DRIVE_SFT_PATH, SFT_PATH)
elif not SFT_PATH.exists():
    from training.generate_sft_data import generate
    print(f'Regenerating SFT data ({CFG.sft_episodes} episodes)...')
    stats = generate(n_episodes=CFG.sft_episodes, out_path=SFT_PATH, episode_length=50)
    print(stats)
else:
    print(f'SFT data already present at {SFT_PATH}')

## 5. Load Qwen2.5-3B in 4-bit with LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 1280

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.float16,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded.')

## 6. SFT warm-start

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files=str(SFT_PATH), split='train')

def to_chat(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}
ds = ds.map(to_chat, remove_columns=ds.column_names)

sft_config = SFTConfig(
    output_dir=f'{DRIVE_RUN_DIR}/sft_out',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=CFG.sft_steps,
    learning_rate=2e-4,
    logging_steps=25,
    fp16=True,
    max_seq_length=MAX_SEQ_LEN,
    report_to=('wandb' if WANDB_ENABLED else 'none'),
    run_name=f'sft-{CFG.name}',
    save_strategy='no',
)
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=sft_config, dataset_text_field='text',
)
sft_trainer.train()
print('SFT done.')

## 7. Build GRPO prompts (curriculum-aware + signal-sum metadata)

Key changes vs Stage 1 prompt collector:
1. Difficulty per episode comes from `cfg.scheduler(step, rng)` — `medium` for `no_curriculum`, ramped for the others.
2. Each prompt now carries `signal_sum` so the GRPO reward function can apply the aux bonus when enabled.

In [ ]:
import random
from datasets import Dataset
from market_env.bots import MarketMakerBot, RandomBot
from market_env.environment import MarketEnvironment
from training.prompts import SYSTEM_PROMPT
from training.rollout import bot_policy, run_episode

PROMPTS_DATASET_SIZE = 1500
scheduler = make_difficulty_scheduler(CFG)

def collect_prompts(n: int) -> list[dict]:
    env = MarketEnvironment()
    rng = random.Random(0)
    out = []
    seed = 0
    step = 0
    while len(out) < n:
        diff = scheduler(step, rng)
        bot = (RandomBot('agent_1', seed=seed) if rng.random() < 0.5
               else MarketMakerBot('agent_1', seed=seed))
        traj = run_episode(env, bot_policy(bot), seed=seed,
                           difficulty=diff, episode_length=50)
        # Per-agent signal sum for the aux reward
        scenario = env._episodes[traj.episode_id].scenario  # noqa: SLF001
        signals = scenario.agent_signals.get('agent_1', {})
        signal_sum = float(sum(signals.values()))
        for i in range(0, len(traj.turns), 5):
            out.append({
                'prompt': tokenizer.apply_chat_template(
                    [{'role': 'system', 'content': SYSTEM_PROMPT},
                     {'role': 'user', 'content': traj.turns[i].prompt}],
                    tokenize=False, add_generation_prompt=True,
                ),
                'true_value': traj.true_value,
                'signal_sum': signal_sum,
            })
            if len(out) >= n:
                break
        seed += 1
        step += 1
    return out

prompts = collect_prompts(PROMPTS_DATASET_SIZE)
prompt_ds = Dataset.from_list(prompts)
print(f'Built {len(prompts)} GRPO prompts. Difficulty mix:')
from collections import Counter
print('  prompt count by difficulty unknown (recorded per-episode), check W&B for verification.')

## 8. GRPO with aux-aware reward function

When `cfg.aux_direction_weight > 0`, buy/sell rewards are nudged up by `min(|signal_sum|/5, 1) * weight` if the trade matches the signal direction. This is the M7B intervention designed to chip at the Probe 3 'say above' bias.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.prompts import parse_action

HOLD_PENALTY = -0.05
CANCEL_BONUS = 0.02
PARSE_FAIL_PENALTY = -1.0
PNL_SCALE_GRPO = 5.0
AUX_WEIGHT = float(CFG.aux_direction_weight)
AUX_REF = 5.0

def oracle_reward(prompts, completions, true_value, signal_sum, **_):
    rewards = []
    for completion, tv, ss in zip(completions, true_value, signal_sum):
        text = completion if isinstance(completion, str) else completion[0]
        action, ok = parse_action(text)
        if not ok:
            rewards.append(PARSE_FAIL_PENALTY); continue
        if action.action_type == 'hold':
            rewards.append(HOLD_PENALTY); continue
        if action.action_type == 'cancel':
            rewards.append(CANCEL_BONUS); continue
        if action.price is None:
            rewards.append(PARSE_FAIL_PENALTY); continue
        side_sign = 1 if action.action_type == 'buy' else -1
        if action.action_type == 'buy':
            r = (tv - action.price) / PNL_SCALE_GRPO
        else:
            r = (action.price - tv) / PNL_SCALE_GRPO
        if AUX_WEIGHT > 0 and ss != 0 and (ss > 0) == (side_sign > 0):
            r += AUX_WEIGHT * min(abs(ss) / AUX_REF, 1.0)
        rewards.append(max(-1.0, min(1.0, r)))
    return rewards

FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir=f'{DRIVE_RUN_DIR}/grpo_checkpoints',
    max_steps=CFG.grpo_steps,
    per_device_train_batch_size=4,
    num_generations=4,
    learning_rate=5e-6,
    logging_steps=CFG.grpo_logging_steps,
    save_steps=CFG.grpo_save_steps,
    save_total_limit=2,
    fp16=True,
    max_prompt_length=1024,
    max_completion_length=80,
    temperature=CFG.temperature,
    report_to=('wandb' if WANDB_ENABLED else 'none'),
    run_name=f'grpo-{CFG.name}',
)
grpo_trainer = GRPOTrainer(
    model=model, reward_funcs=oracle_reward,
    args=grpo_config, train_dataset=prompt_ds,
)
grpo_trainer.train()
grpo_trainer.save_model(f'{DRIVE_RUN_DIR}/grpo_final')
print(f'GRPO done. Adapter saved to {DRIVE_RUN_DIR}/grpo_final')

## 9. Eval + ToM probes for this preset

Saves results to `training/runs/<preset_name>/` so `notebooks/analysis.ipynb` picks them up automatically.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from market_env.environment import MarketEnvironment
from training.evaluate import run_evaluation, save_summary
from training.tom_probes import (
    direction_inference, price_efficiency, run_probe_episode, signal_alignment,
)

FastLanguageModel.for_inference(model)

OUT = Path(f'training/runs/{CFG.name}')
OUT.mkdir(parents=True, exist_ok=True)

def _generate(messages, max_new_tokens=80, temperature=0.7):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=temperature > 0, temperature=max(temperature, 1e-5),
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def llm_policy(obs):
    from training.prompts import SYSTEM_PROMPT, format_observation, parse_action
    text = _generate([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': format_observation(obs)},
    ])
    return parse_action(text)

def llm_answer_fn(prompt: str) -> str:
    """Probe 3 helper: free-text answer for the direction-inference probe."""
    return _generate(
        [{'role': 'system', 'content': 'You are a careful trader.'},
         {'role': 'user', 'content': prompt}],
        max_new_tokens=8, temperature=0.0,
    )

# ---- Standard eval ----
summary = run_evaluation(
    llm_policy, label=f'trained_{CFG.name}', seeds=list(CFG.eval_seeds),
    difficulty=CFG.eval_difficulty, bot_config=CFG.eval_bot_config,
)
save_summary(summary, OUT / 'eval_trained.json')
print(f'Mean P&L: {summary.mean_pnl_normalized:+.4f}, '
      f'parse: {summary.parse_success_rate:.0%}, '
      f'active: {summary.participation_rate:.0%}')

# ---- ToM Probes 1 & 2 ----
probe_trajs = []
for seed in range(100, 110):
    env = MarketEnvironment()
    probe_trajs.append(run_probe_episode(env, llm_policy, seed=seed, difficulty='medium'))
pe = asdict(price_efficiency(probe_trajs, label='trained'))
sa = asdict(signal_alignment(probe_trajs, label='trained'))
(OUT / 'tom_probes_trained.json').write_text(
    json.dumps({'trained': {'price_efficiency': pe, 'signal_alignment': sa}}, indent=2),
    encoding='utf-8',
)
print(f'Probe 1 final price err: ${pe["final_mean_error"]:.2f} '
      f'(improvement {pe["improvement"]:+.2f})')
print(f'Probe 2 alignment rate: {sa["alignment_rate"]:.0%} '
      f'(n_active={sa["n_active"]})')

# ---- ToM Probe 3 ----
di = asdict(direction_inference(
    answer_fn=llm_answer_fn, label=f'trained_{CFG.name}', n_probes=20,
))
(OUT / 'tom_direction_inference.json').write_text(json.dumps(di, indent=2), encoding='utf-8')
print(f'Probe 3 accuracy: {di["accuracy"]:.0%} ({di["n_correct"]}/{di["n_probes"]} correct)')
print(f'\nAll artefacts written to {OUT.resolve()}')

## 10. Push results back to your local repo

Download the three JSON files from `training/runs/<preset_name>/` (left sidebar → file browser → right-click → Download), drop them into the same path on your laptop, then locally run:

```
python -m training.results_matrix --runs-root training/runs --markdown training/runs/results_matrix.md
jupyter nbconvert --to notebook --execute notebooks/analysis.ipynb --inplace
```

That regenerates every plot + the comparison table.

In [ ]:
# Convenience: tar up just the result JSONs for one-click download.
import shutil
shutil.make_archive(f'/content/{CFG.name}_results', 'zip', root_dir=str(OUT))
print(f'Wrote /content/{CFG.name}_results.zip — download via the file browser.')